<a href="https://colab.research.google.com/github/Madankk-06/Deep-learning-projects/blob/main/3_Vectorized_Convolutional_Forward_Pass_via_im2col.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Import NumPy for implementing vectorized convolution operations.
import numpy as np

# Import SciPy for optional padding and array utilities.
from scipy import signal

In [2]:
# Convert the input feature map into a 2D matrix.
# The im2col transformation rearranges every receptive field
# into one column, allowing convolution to be computed as
# a single matrix multiplication.

def im2col(input_data, kernel_h, kernel_w, stride=1, padding=0):

    # Apply zero padding around the input feature map.
    input_padded = np.pad(
        input_data,
        ((0,0),(0,0),(padding,padding),(padding,padding)),
        mode='constant'
    )

    N, C, H, W = input_padded.shape

    # Compute output feature map dimensions.
    out_h = (H - kernel_h)//stride + 1
    out_w = (W - kernel_w)//stride + 1

    cols = []

    # Extract every sliding window from the input.
    for y in range(out_h):
        for x in range(out_w):

            patch = input_padded[
                :,
                :,
                y*stride:y*stride+kernel_h,
                x*stride:x*stride+kernel_w
            ]

            # Flatten every receptive field into a column vector.
            cols.append(patch.reshape(N,-1))

    # Stack all extracted patches into a unified 2D matrix.
    cols = np.stack(cols, axis=1)

    return cols

In [3]:
# Initialize learnable convolution filters.
# Every filter learns a unique spatial feature.

def initialize_filters(out_channels, in_channels, kernel_h, kernel_w):

    filters = np.random.randn(
        out_channels,
        in_channels,
        kernel_h,
        kernel_w
    )

    return filters

In [4]:
# Perform the forward convolution using matrix multiplication.
# The convolution operation is transformed into:
#
# Output = Filter Matrix × im2col Matrix

def convolution_forward(input_data, filters, stride=1, padding=0):

    out_channels, in_channels, kernel_h, kernel_w = filters.shape

    # Convert input image into column representation.
    input_cols = im2col(
        input_data,
        kernel_h,
        kernel_w,
        stride,
        padding
    )

    # Flatten every convolution filter into one row.
    filter_matrix = filters.reshape(
        out_channels,
        -1
    )

    N = input_data.shape[0]

    output = []

    # Perform vectorized convolution using dot product.
    for i in range(N):

        conv = np.dot(
            input_cols[i],
            filter_matrix.T
        )

        output.append(conv)

    output = np.array(output)

    return output

In [5]:
# Create a batch of input feature maps.
# Shape:
# Batch Size = 2
# Channels = 3
# Height = 32
# Width = 32

input_tensor = np.random.randn(
    2,
    3,
    32,
    32
)

In [6]:
# Create convolution filters.
# Output Channels = 16
# Input Channels = 3
# Kernel Size = 3 × 3

filters = initialize_filters(
    16,
    3,
    3,
    3
)

In [7]:
# Compute the vectorized convolution output.
# Padding preserves border information,
# while stride controls the movement of the kernel.

output = convolution_forward(
    input_tensor,
    filters,
    stride=1,
    padding=1
)

In [8]:
# Verify the output dimensions generated
print("Output Shape :", output.shape)

Output Shape : (2, 1024, 16)
